# 05 — Exploration des tables gold et choix des graphiques

Ce notebook explore les cinq tables gold (PostgreSQL) pour répondre aux questions de la directrice des
opérations, et choisit, graphique par graphique, la forme qui sera reprise dans le dashboard. Il se termine par un
**avant / après** : le camembert des causes de retard refait selon les principes de perception.

**pandas uniquement sur les tables gold** : elles comptent de 168 à 6 805 lignes. Les 7 millions de vols restent
dans le data lake, agrégés par Spark dans `03_gold`.

**Règles appliquées à chaque graphique (séance S04)**

| Règle | Pourquoi |
|---|---|
| Le titre énonce la **conclusion**, le sous-titre dit ce qui est mesuré | le lecteur pressé lit le titre et s'arrête là |
| Unités sur les axes, barres qui partent de 0 | une barre code par sa longueur : tronquer l'axe exagère les écarts |
| Pas de camembert, pas d'arc-en-ciel | on compare mal les angles et les aires, et des couleurs sans ordre ne portent aucun sens |
| **Une seule couleur d'accent**, et seulement si elle a un sens | ici le bleu veut toujours dire « là où agir » ; le reste est gris |
| Étiquettes directes plutôt que légende | pas d'aller-retour de l'œil entre la légende et les marques |

Les titres sont **calculés à partir des données** : relancé sur d'autres données, le notebook garde des titres
vrais.

## 1. Lecture des tables et style commun

**Ré-agréger à partir des comptes.** Dès qu'on regroupe plusieurs lignes (plusieurs mois, plusieurs compagnies),
on recalcule les taux à partir des comptes (`somme des vols à l'heure / somme des vols opérés`), jamais en
faisant la moyenne des pourcentages : un mois de 120 000 vols pèse plus qu'un mois de 6 000. Les tables de
`03_gold` contiennent ces comptes (`vols_operes`, `vols_a_l_heure`, `vols_annules`) ; si une table ne les a
pas (contrat strict), `avec_comptes()` les reconstitue à partir des pourcentages.

In [ ]:
import sys
sys.path.append("/home/jovyan/work")
from commun.donnees import lire

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter

# Palette : une seule couleur d'accent, qui veut toujours dire « là où agir » ; le reste en gris
ACCENT = "#257de0"
GRIS = "#b4b4b4"
ENCRE, ENCRE_2, GRILLE, AXE = "#191919", "#666666", "#e7e7e7", "#c2c2c2"
SEQUENTIEL = LinearSegmentedColormap.from_list(   # une seule teinte, du clair (peu) au foncé (beaucoup)
    "bleus", ["#cce3fd", "#9dc8f7", "#6aabf1", "#338ced", "#2370cd", "#1757ac", "#0e3d7e"])

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.edgecolor": AXE, "axes.labelcolor": ENCRE_2, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlecolor": ENCRE, "axes.spines.top": False,
    "axes.spines.right": False, "grid.color": GRILLE, "grid.linewidth": 0.8,
    "xtick.color": ENCRE_2, "ytick.color": ENCRE_2, "text.color": ENCRE, "font.size": 10,
})
MOIS = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août", "septembre", "octobre",
        "novembre", "décembre"]
JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
EN_PCT = FuncFormatter(lambda v, _: f"{v:.0f} %")


def pct(x, d=1):
    """74.83 → « 74,8 % » (virgule décimale)."""
    return f"{x:.{d}f}".replace(".", ",") + " %"


def milliers(n):
    return f"{int(round(n)):,}".replace(",", " ")


def titrer(ax, conclusion, mesure):
    """Titre = la conclusion ; sous-titre gris = ce qui est mesuré."""
    ax.set_title(conclusion, pad=26)
    ax.text(0, 1.02, mesure, transform=ax.transAxes, color=ENCRE_2, fontsize=9.5, va="bottom")


def avec_comptes(df, vols="vols"):
    """Ajoute vols_operes / vols_a_l_heure / vols_annules s'ils manquent (approximation : les détournés,
    ≈ 0,25 % des vols, restent alors comptés comme opérés)."""
    df = df.copy()
    if "vols_operes" not in df:
        df["vols_annules"] = df[vols] * df["pct_annules"] / 100
        df["vols_operes"] = df[vols] - df["vols_annules"]
        df["vols_a_l_heure"] = df["vols_operes"] * df["pct_a_l_heure"] / 100
    return df


ponctualite = avec_comptes(lire("gold_ponctualite_mensuelle"))
causes = lire("gold_causes_retard")
aeroports = avec_comptes(lire("gold_aeroports"), vols="departs")
heure_jour = lire("gold_heure_jour")
routes = lire("gold_routes")
ponctualite["mois"] = pd.to_datetime(ponctualite["mois"])

for nom, table in [("gold_ponctualite_mensuelle", ponctualite), ("gold_causes_retard", causes),
                   ("gold_aeroports", aeroports), ("gold_heure_jour", heure_jour), ("gold_routes", routes)]:
    print(f"{nom:<27} {len(table):>5} lignes")

NATIONAL = 100 * ponctualite.vols_a_l_heure.sum() / ponctualite.vols_operes.sum()
print(f"\nPonctualité nationale 2024 : {pct(NATIONAL)} des vols opérés arrivés à l'heure "
      f"({milliers(ponctualite.vols.sum())} vols programmés)")

## 2. Contexte : la ponctualité au fil de l'année

*Les retards sont-ils saisonniers ?* (sous-question 2 du persona). Une **ligne**, car l'œil y lit une évolution
dans le temps. La ligne code par la **position** et non par la longueur : son axe n'a pas à partir de 0
(Cleveland & McGill). On garde une plage d'une trentaine de points pour ne pas grossir les écarts, et la moyenne
annuelle sert de repère, étiquetée directement.

In [ ]:
mensuel = ponctualite.groupby("mois")[["vols", "vols_operes", "vols_a_l_heure"]].sum()
mensuel["pct"] = 100 * mensuel.vols_a_l_heure / mensuel.vols_operes
creux, pic = mensuel.pct.idxmin(), mensuel.pct.idxmax()

fig, ax = plt.subplots(figsize=(10, 4.4))
ax.plot(mensuel.index, mensuel.pct, color=ACCENT, lw=2, marker="o", ms=6, mec="white", mew=1.5)
ax.axhline(NATIONAL, color=ENCRE_2, lw=0.8)
# Étiquette de la moyenne dans la marge droite, hors de la courbe
ax.annotate(f"moyenne\n{pct(NATIONAL)}", xy=(1, NATIONAL), xycoords=("axes fraction", "data"), xytext=(6, 0),
            textcoords="offset points", va="center", color=ENCRE_2, fontsize=9)
for date, place in [(creux, "top"), (pic, "bottom")]:
    decalage = -1.0 if place == "top" else 1.0
    ax.annotate(f"{MOIS[date.month - 1]} : {pct(mensuel.pct[date])}", (date, mensuel.pct[date]),
                xytext=(0, 12 * decalage), textcoords="offset points", ha="center", va=place, fontsize=9.5,
                fontweight="bold")

bas = 5 * np.floor(mensuel.pct.min() / 5) - 10
ax.set_ylim(bas, max(bas + 30, 5 * np.ceil(mensuel.pct.max() / 5) + 5))
ax.yaxis.set_major_formatter(EN_PCT)
ax.set_ylabel("Vols arrivés à l'heure (%)")
ABREGES = ["janv.", "févr.", "mars", "avr.", "mai", "juin", "juil.", "août", "sept.", "oct.", "nov.", "déc."]
ax.set_xticks(mensuel.index, [ABREGES[d.month - 1] for d in mensuel.index])
ax.grid(axis="y")
ecart = mensuel.pct[pic] - mensuel.pct[creux]
titrer(ax, f"La ponctualité varie de {ecart:.0f} points dans l'année : creux en {MOIS[creux.month - 1]}, "
           f"pic en {MOIS[pic.month - 1]}",
       "Part des vols opérés arrivés avec moins de 15 min de retard, toutes compagnies, par mois de 2024")
plt.tight_layout()
plt.show()

**Lecture.** Les creux tombent sur les mois de forte demande et de météo difficile (orages d'été, tempêtes
d'hiver) : c'est là que la directrice doit renforcer les marges. **Dans le dashboard** : même courbe, en
« Contexte », recalculée pour la compagnie et la période choisies, la moyenne de toutes les compagnies servant
de référence grise.

## 3. Les compagnies face à la moyenne nationale

*Comment se compare notre compagnie ?* (sous-question 5). Des **barres horizontales** (les noms se lisent sans
pivoter), **triées** pour que le rang se lise sans effort, **à partir de 0**. Le bleu marque les compagnies sous la
moyenne nationale ; la valeur est écrite au bout de chaque barre.

In [ ]:
cie = (ponctualite.groupby("compagnie_nom")[["vols", "vols_operes", "vols_a_l_heure"]].sum()
       .assign(pct=lambda d: 100 * d.vols_a_l_heure / d.vols_operes).sort_values("pct"))
sous = cie[cie.pct < NATIONAL]

fig, ax = plt.subplots(figsize=(9.5, 5.6))
ax.barh(cie.index, cie.pct, color=np.where(cie.pct < NATIONAL, ACCENT, GRIS), height=0.72)
for y, v in enumerate(cie.pct):
    ax.text(v + 1, y, pct(v), va="center", fontsize=9)
ax.axvline(NATIONAL, color=ENCRE, lw=0.8)
ax.text(NATIONAL, len(cie) - 0.3, f" moyenne nationale {pct(NATIONAL)}", fontsize=9, va="bottom")
ax.set_xlim(0, 100)
ax.xaxis.set_major_formatter(EN_PCT)
ax.set_xlabel("Vols arrivés à l'heure (%)")
ax.tick_params(axis="y", length=0)
titrer(ax, f"{len(sous)} compagnies sur {len(cie)} sont sous la moyenne nationale ; "
           f"{cie.index[0]} ferme la marche ({pct(cie.pct.iloc[0])})",
       "Vols opérés arrivés avec moins de 15 min de retard, 2024. En bleu : sous la moyenne nationale.")
plt.tight_layout()
plt.show()

## 4. Les aéroports : où sont les points noirs ?

*Quels aéroports concentrent les retards ?* (sous-question 1). On garde les **20 aéroports qui ont le plus de
départs** : un petit aéroport à 60 % pèse peu dans le réseau, un grand hub à 72 % désorganise des centaines de
correspondances. Même forme et même code couleur que pour les compagnies. Dans le dashboard, ces aéroports sont
placés sur une **carte** (dimension géographique), où la taille du point suit le nombre de départs.

In [ ]:
gros = aeroports.nlargest(20, "departs").assign(
    pct=lambda d: 100 * d.vols_a_l_heure / d.vols_operes,
    nom=lambda d: d.code_iata + " · " + d.ville).sort_values("pct")
noirs = gros[gros.pct < NATIONAL].sort_values("pct")

fig, ax = plt.subplots(figsize=(9.5, 6.4))
ax.barh(gros.nom, gros.pct, color=np.where(gros.pct < NATIONAL, ACCENT, GRIS), height=0.72)
for y, (v, d) in enumerate(zip(gros.pct, gros.departs)):
    ax.text(v + 1, y, f"{pct(v)}   ({milliers(d)} départs)", va="center", fontsize=8.5)
ax.axvline(NATIONAL, color=ENCRE, lw=0.8)
ax.text(NATIONAL, len(gros) - 0.3, f" moyenne nationale {pct(NATIONAL)}", fontsize=9, va="bottom")
ax.set_xlim(0, 115)
ax.set_xticks(range(0, 101, 20))
ax.xaxis.set_major_formatter(EN_PCT)
ax.set_xlabel("Départs arrivés à l'heure (%)")
ax.tick_params(axis="y", length=0)
titrer(ax, f"{len(noirs)} des 20 plus gros aéroports sont sous la moyenne, "
           f"{', '.join(noirs.code_iata[:3])} en tête des points noirs",
       "Départs 2024 arrivés avec moins de 15 min de retard, aéroports classés par départs. En bleu : sous la moyenne.")
plt.tight_layout()
plt.show()

## 5. Les créneaux à risque : heatmap jour × heure

*Quels jours et quelles heures sont les plus critiques ?* (sous-question 2). Deux dimensions ordonnées (jour,
heure) et une grandeur : une **heatmap**. La couleur code ici une **quantité**, donc une seule teinte, du clair (peu
de retards) au foncé (beaucoup), avec une échelle qui part de 0 et une légende graduée en %. Pas d'arc-en-ciel :
ses teintes n'ont pas d'ordre naturel et créent des frontières qui n'existent pas.

Les créneaux de nuit qui comptent moins de 0,05 % des vols sont laissés en blanc : sur si peu de vols, le taux
n'est pas fiable. Seules les deux cases extrêmes sont étiquetées.

In [ ]:
taux = heure_jour.pivot(index="jour_semaine", columns="heure", values="pct_retard")
volume = heure_jour.pivot(index="jour_semaine", columns="heure", values="vols")
fiable = volume >= 0.0005 * heure_jour.vols.sum()
taux_affiche = taux.where(fiable)

pire = taux_affiche.stack().idxmax()
meilleur = taux_affiche.stack().idxmin()
matin = heure_jour[heure_jour.heure.between(6, 9)]
soir = heure_jour[heure_jour.heure.between(17, 21)]
moyenne = lambda d: (d.pct_retard * d.vols).sum() / d.vols.sum()

fig, ax = plt.subplots(figsize=(11, 3.9))
im = ax.imshow(taux_affiche, cmap=SEQUENTIEL, vmin=0, vmax=5 * np.ceil(taux_affiche.max().max() / 5), aspect="auto")
ax.set_yticks(range(7), [j.capitalize() for j in JOURS])
ax.set_xticks(range(0, 24, 2), [f"{h} h" for h in range(0, 24, 2)])
ax.set_xlabel("Heure de départ prévue (heure locale)")
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
for (jour, heure), texte_couleur in [(pire, "white"), (meilleur, ENCRE)]:
    ax.text(taux.columns.get_loc(heure), jour - 1, f"{taux.loc[jour, heure]:.0f} %", ha="center", va="center",
            fontsize=8.5, fontweight="bold", color=texte_couleur)
barre = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, format=EN_PCT)
barre.set_label("Vols en retard (%)", color=ENCRE_2)
barre.outline.set_visible(False)

if moyenne(soir) > moyenne(matin) + 3:
    conclusion = (f"Les retards s'accumulent au fil de la journée : {pct(moyenne(soir), 0)} des vols du soir "
                  f"(17-21 h) en retard, contre {pct(moyenne(matin), 0)} le matin (6-9 h)")
else:
    conclusion = f"L'heure de départ change peu le risque de retard ({pct(moyenne(matin), 0)} le matin, {pct(moyenne(soir), 0)} le soir)"
titrer(ax, conclusion, f"Vols opérés arrivés avec 15 min de retard ou plus, 2024. Pire créneau : "
                       f"{JOURS[pire[0] - 1]} {pire[1]} h ; meilleur : {JOURS[meilleur[0] - 1]} {meilleur[1]} h.")
plt.tight_layout()
plt.show()

**Lecture.** Le retard se propage : un avion en retard le matin le reste sur ses rotations suivantes (c'est la
cause « avion en retard », la première en minutes, voir la section 7). Les premiers départs du matin sont les
plus sûrs. **Levier** : protéger les rotations de fin de journée (marges au sol, avions de réserve).

## 6. Les routes à traiter en premier

*Quelles routes présentent le pire ratio ponctualité / volume ?* (sous-question 4). Le **score de priorité**
(`vols × (1 − pct_a_l_heure / 100)`, défini dans le contrat gold) est environ le nombre de vols en retard sur
l'année : il combine volume et taux. Une seule série, donc une seule couleur ; chaque barre porte son score et
son taux de ponctualité.

In [ ]:
top = routes.nlargest(10, "score_priorite").sort_values("score_priorite")
top["nom"] = top.origine + " → " + top.destination

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.barh(top.nom, top.score_priorite, color=ACCENT, height=0.7)
for y, (s, p) in enumerate(zip(top.score_priorite, top.pct_a_l_heure)):
    ax.text(s + top.score_priorite.max() * 0.01, y, f"{milliers(s)}  ·  {pct(p)} à l'heure", va="center",
            fontsize=9)
ax.set_xlim(0, top.score_priorite.max() * 1.3)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: milliers(v)))
ax.set_xlabel("Score de priorité ≈ vols en retard sur l'année")
ax.tick_params(axis="y", length=0)
premiere = top.iloc[-1]
titrer(ax, f"10 routes cumulent {milliers(top.score_priorite.sum())} vols en retard par an ; "
           f"{premiere.nom} en tête ({milliers(premiere.score_priorite)})",
       "Score de priorité = vols × (1 − part à l'heure), 2024, les 10 routes au plus fort score")
plt.tight_layout()
plt.show()

## 7. Avant / après : les causes de retard

*Quelle est la part de chaque cause, et sur lesquelles peut-on agir ?* (sous-question 3). On part de la même
donnée, la somme des minutes de retard par cause sur l'année (`gold_causes_retard` : on somme les minutes, puis
on divise par le total ; jamais la moyenne des `part_pct`).

### Avant : le camembert par défaut

In [ ]:
par_cause = (causes.groupby(["cause", "controlable"], as_index=False)["minutes"].sum()
             .assign(part=lambda d: 100 * d.minutes / d.minutes.sum()))

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(par_cause.minutes, labels=par_cause.cause, autopct="%1.1f%%")
ax.set_title("Causes de retard", loc="center")
plt.show()

**Ce qui ne va pas**
1. **Angles et aires.** On compare mal des angles : 34 % et 41 % se ressemblent, et la tranche « securite »
   (0,2 %) est invisible. Cleveland & McGill classent l'angle et l'aire loin derrière la position et la longueur
   pour juger une quantité.
2. **Cinq couleurs sans signification.** La palette par défaut donne une teinte par cause, sans ordre ni sens. La
   seule distinction utile à la directrice, **contrôlable ou non**, n'est pas codée.
3. **Aucun ordre.** Les tranches suivent l'ordre alphabétique des codes : il faut chercher la plus grosse.
4. **Titre descriptif.** « Causes de retard » dit ce qu'on regarde, pas ce qu'il faut en retenir.
5. **Libellés techniques** (`avion_retard`, `nas`), pourcentages tassés et qui se chevauchent sur les petites
   tranches.

### Après : barres horizontales triées, une seule couleur d'accent

In [ ]:
LIBELLES = {
    "avion_retard": "Avion en retard (vol précédent)",
    "compagnie": "Compagnie (équipage, maintenance, sol)",
    "nas": "Système national (contrôle aérien, trafic)",
    "meteo": "Météo extrême",
    "securite": "Sécurité",
}
# Les causes contrôlables d'abord, puis tri décroissant dans chaque groupe (ici, c'est aussi l'ordre global)
apres = par_cause.sort_values(["controlable", "part"], ascending=[False, False]).reset_index(drop=True)
part_controlable = apres.loc[apres.controlable, "part"].sum()

fig, ax = plt.subplots(figsize=(10, 4.2))
y = np.arange(len(apres))
ax.barh(y, apres.part, color=np.where(apres.controlable, ACCENT, GRIS), height=0.68)
ax.set_yticks(y, [LIBELLES.get(c, c) for c in apres.cause])
ax.invert_yaxis()                                     # la plus grande barre en haut
for yi, v in zip(y, apres.part):
    ax.text(v + 0.6, yi, pct(v), va="center", fontsize=9.5)

# Étiquette directe des deux groupes (à la place d'une légende) : une accolade et le total du groupe
x_groupe = apres.part.max() + 9
for controlable, nom in [(True, "Contrôlables"), (False, "Non contrôlables")]:
    lignes = y[apres.controlable == controlable]
    if len(lignes):
        ax.plot([x_groupe, x_groupe], [lignes.min() - 0.3, lignes.max() + 0.3], color=ENCRE_2, lw=1)
        total = apres.loc[apres.controlable == controlable, "part"].sum()
        ax.text(x_groupe + 1, lignes.mean(), f"{nom}\n{pct(total)}", va="center", fontsize=10,
                fontweight="bold" if controlable else "normal", color=ENCRE if controlable else ENCRE_2)

ax.set_xlim(0, x_groupe + 16)
ax.set_xticks(range(0, int(apres.part.max()) + 10, 10))
ax.xaxis.set_major_formatter(EN_PCT)
ax.set_xlabel("Part des minutes de retard (%)")
ax.tick_params(axis="y", length=0)
titrer(ax, f"{pct(part_controlable)} des minutes de retard ont une cause que la compagnie contrôle",
       "Minutes de retard par cause, toutes compagnies, 2024. En bleu : causes contrôlables par la compagnie.")
plt.tight_layout()
plt.show()

### Chaque correction et sa justification

| Correction | Principe | Effet pour la lectrice |
|---|---|---|
| Camembert → **barres horizontales** | **Cleveland & McGill** : on juge une quantité bien plus justement par une position ou une longueur sur une échelle commune que par un angle ou une aire | 41 % et 34 % se départagent d'un coup d'œil ; les 0,2 % de la sécurité restent visibles |
| **Tri décroissant**, causes contrôlables groupées en haut | l'ordre spatial porte le classement : pas besoin de comparer pour trouver la plus grande | la cause n° 1 est lue en premier |
| **Axe à 0** | une barre code par sa longueur : un axe tronqué fausse le rapport entre les barres | 40 % paraît deux fois plus long que 20 % |
| **Une couleur d'accent** pour les causes contrôlables, **le reste en gris** | **couleur = sens** : la couleur ne sert que si elle encode une information, ici la seule qui compte pour décider | le regard va droit aux leviers d'action ; cinq teintes arbitraires disparaissent |
| **Étiquettes directes** (valeur au bout des barres, accolade nommant chaque groupe et son total) | supprimer l'aller-retour entre la légende et les marques | la part contrôlable (≈ trois quarts) se lit sans calcul |
| Fond, cadre et grille supprimés, axes discrets | **ratio données / encre** (Tufte) : chaque trait doit porter de l'information | l'attention reste sur les barres |
| **Titre = la conclusion**, sous-titre = la mesure | le titre est ce que l'on retient | le message est donné, pas à deviner |
| Libellés en clair (« Avion en retard (vol précédent) ») | le vocabulaire de la lectrice, pas celui de la base | pas de code à décoder |

**Pour la directrice** : la cause n° 1 est l'avion qui arrive en retard de son vol précédent, et c'est un levier
qu'elle tient (marges de rotation, avions de réserve). Ce graphique est repris tel quel dans la partie
« Constat » du dashboard, recalculé selon les filtres.

## Ce que le dashboard reprend

| Partie du récit | Graphique | Table gold |
|---|---|---|
| Contexte | 4 KPI + courbe mensuelle de ponctualité | `gold_ponctualite_mensuelle`, `gold_causes_retard` |
| Constat | causes en barres (version « après »), carte des aéroports, heatmap jour × heure | `gold_causes_retard`, `gold_aeroports`, `gold_heure_jour` |
| Recommandation | top routes par score de priorité + une phrase d'action | `gold_routes` |

## C'est réussi quand…
- chaque titre énonce une conclusion vraie sur les données affichées ;
- aucune barre ne part d'ailleurs que de 0, aucun graphique n'utilise plus d'une couleur d'accent ;
- l'avant / après montre le même chiffre sous les deux formes, et chaque correction est justifiée.